# M03 Immunoinformatics: finding and ranking T cell epitopes with AI

**ISCB-LATAM 2026, Lima. Tutorial IP3.**

From a protein sequence to a prioritised shortlist of epitope candidates.

**Case study: FhaC**, the filamentous hemagglutinin transporter of *Bordetella pertussis* (UniProtKB P35077, 584 aa). The structural module folds the same protein with AlphaFold, so the two tell one story.

---

## The five questions

| Slide | Question | What you will do here |
|---|---|---|
| 18, 19 | Where does the protein go? | Score DeepTMHMM against curated annotation |
| 20, 21 | Which peptides can MHC bind? | Run **NetMHCpan and NetMHCIIpan with no licence**, then cross three tools |
| 22 | Which are immunogenic? | Rebuild DeepImmuno-CNN and the PRIME input vector |
| 23 | Which TCR recognises them? | Rebuild NetTCR-2.0, then break it on purpose |
| 24 | Which do we take forward? | Chain pVACtools style filters into a shortlist |

**No licensed software is needed for any of this.** CPU is enough, nothing needs a GPU. Budget 30 to 40 minutes.

Every arrow narrows the list. No arrow proves a T cell response.

## 0. Setup

The slowest cell in the notebook, around 5 minutes. Run it once per session.

In [ ]:
%%capture
!pip install -q mhcflurry biopython
!mhcflurry-downloads fetch models_class1_presentation

In [ ]:
# Getting the module into Colab.
#
# The tutorial repository is PRIVATE, so a plain clone fails. Three ways in,
# tried in order automatically:
#
#   A. Paste a GitHub token below. Create one at github.com/settings/tokens
#      with 'repo' scope. Use this while the repo stays private.
#   B. Leave TOKEN empty once the repository is made public.
#   C. Neither: the cell asks you to upload a zip of the module folder.
#      Get it from GitHub with the green Code button, Download ZIP.

REPO = "BigMindLab/Tutorial_IA_ISCB-LATAM_2026"
BRANCH = "m03-immunoinformatics-exercises"
TOKEN = ""  # paste a GitHub token here, or leave empty

import os, pathlib, shutil, subprocess, zipfile

MODULE = pathlib.Path("/content/M03_immunoinformatics")

def try_clone():
    root = pathlib.Path("/content/tutorial_repo")
    if root.exists():
        shutil.rmtree(root)
    url = (f"https://{TOKEN}@github.com/{REPO}.git" if TOKEN
           else f"https://github.com/{REPO}.git")
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        url, str(root)], capture_output=True, text=True)
    src = root / "modules" / "M03_immunoinformatics"
    if src.exists():
        shutil.move(str(src), str(MODULE))
        return True
    print("clone failed:", (r.stderr or "").strip()[:300])
    return False

def ask_for_zip():
    from google.colab import files
    print("Upload a zip that contains the M03_immunoinformatics folder.")
    for name in files.upload():
        with zipfile.ZipFile(name) as z:
            z.extractall("/content/unzipped")
    for found in pathlib.Path("/content/unzipped").rglob("solutions"):
        if (found.parent / "utils").exists():
            shutil.move(str(found.parent), str(MODULE))
            return True
    print("Could not find the module inside that zip.")
    return False

ok = MODULE.exists() or try_clone() or ask_for_zip()
if not ok:
    raise SystemExit("Module not available, nothing below will run.")

os.chdir(MODULE)
print("working directory:", os.getcwd())
print(sorted(p.name for p in pathlib.Path('.').iterdir()))

In [ ]:
# Fetches the training data from UniProt and from the DeepImmuno and NetTCR
# repositories. This repository does not redistribute third party data.
!python utils/download_data.py

## The inputs

Two kinds, and it is worth keeping them apart.

**The protein.** `data/fhaC.fasta` ships with the module, so there is nothing to do if you are following the case study. The cell below lets you swap in your own sequence instead.

**The training data.** Downloaded by the cell above from UniProt and from the DeepImmuno and NetTCR repositories. It is never used as a target, only to fit the models. Nothing in this module trains on your protein, because a protein with no measured immunogenicity carries no labels to learn from.

**The TCR sequences.** `data/tcrs_example.csv` holds 20 real published CDR3 pairs. This is the one input a protein sequence can never provide, and exercise 13 explains why it matters.

> **Using your own protein.** Run the next cell and pick `upload`. Everything downstream works, with one caveat: the curated UniProt annotation only applies to fhaC. For any other sequence the topology filter falls back to DeepTMHMM, which is a prediction rather than evidence.

In [ ]:
# Choose the protein.
#   "fhaC"   the case study, already in data/fhaC.fasta
#   "upload" pick a FASTA from your computer
#   "paste"  paste a sequence into SEQUENCE below

PROTEIN = "fhaC"
SEQUENCE = ""   # only used when PROTEIN = "paste"
NAME = "my_protein"

import pathlib, shutil

data = pathlib.Path("data")
target = data / "fhaC.fasta"   # every script reads this path

if PROTEIN == "upload":
    from google.colab import files
    up = files.upload()
    src = list(up)[0]
    shutil.copy(src, target)
    print("using uploaded file:", src)

elif PROTEIN == "paste":
    clean = "".join(SEQUENCE.split()).upper()
    if not clean:
        raise ValueError("SEQUENCE is empty")
    target.write_text(">%s\n%s\n" % (NAME, clean))
    print("using pasted sequence")

# Both topology sources describe fhaC and nothing else: the curated UniProt
# entry, and the DeepTMHMM run committed in data/precomputed. For any other
# protein they are simply wrong, so they get set aside.
if PROTEIN != "fhaC":
    for stale in (data / "P35077.json", data / "precomputed" / "deeptmhmm_fhaC.3line"):
        if stale.exists():
            stale.rename(stale.with_suffix(stale.suffix + ".fhaC_only"))
    print("\nfhaC topology set aside, because it does not describe your protein.")
    print("The signal peptide filter is therefore SKIPPED, and peptides that")
    print("would be cleaved off during secretion stay in the ranking.")
    print("To enable it, run SignalP or DeepTMHMM on your own sequence and drop")
    print("the .3line output into data/precomputed/deeptmhmm_fhaC.3line")

header, seq = "", []
for line in target.read_text().splitlines():
    if line.startswith(">"):
        header = line[1:]
    elif line.strip():
        seq.append(line.strip())
seq = "".join(seq)

print("\nprotein :", header[:70])
print("length  :", len(seq), "residues")
print("9-mers  :", len(seq) - 8, "  (%d - 9 + 1)" % len(seq))
print("first 60:", seq[:60])

---
## Exercise 11, slides 18 and 19: where does fhaC go?

Slides 18 and 19 present SignalP 6.0 and DeepTMHMM and end with the same warning: neither one predicts antigenicity. Running them is easy. Knowing whether to believe them is the actual skill, and that needs something to compare against.

This exercise builds that reference from the curated UniProt annotation of P35077, which comes partly from a solved structure. A verified DeepTMHMM run on fhaC is included in the repository, so the comparison works even without installing anything.

> The reference is curated annotation, not ground truth. Where a predictor and UniProt disagree, the interesting question is which one to trust.

In [ ]:
!python solutions/ex11_topology_reference.py --deeptmhmm data/precomputed/deeptmhmm_fhaC.3line

### What to notice

DeepTMHMM predicts **16 beta strands** and UniProt annotates **16**. The signal peptide boundary differs by two residues. Agreement on the strand or not strand question is around 88 percent.

The raw per residue agreement looks much worse, around 47 percent, purely because the reference cannot distinguish periplasm from outside. That gap between two numbers describing the same comparison is the lesson.

### Discuss

1. SignalP gives a type and a cleavage site. What does each one actually constrain downstream?
2. If a predictor disagrees with curated annotation, is the predictor wrong?
3. fhaC is an extracellular bacterial protein. Which MHC class is its natural route, and what does that imply for everything that follows?

---
## Exercises 15 and 16, slides 20 and 21: binding, with no licence

Slides 20 and 21 are built on NetMHCpan 4.1 and NetMHCIIpan 4.0. Both need an academic licence from DTU plus a manual install, which would stop a workshop before it starts.

**IEDB runs the same tools on its own cluster behind a public API.** Exercise 15 sends FhaC there and parses the answer, so you get real NetMHCpan numbers with nothing installed. These are the actual DTU tools, not a stand in.

Exercise 16 then does the part that matters. It crosses three independent class I tools and asks which peptides all of them call a strong binder. MixMHCpred is position weight matrices rather than a neural network, so their agreement is not three versions of the same model nodding at each other.

> **Needs internet, takes a couple of minutes.** The cell sends one request per allele and waits between calls, because the API is a free shared service. Please do not put it in a loop.

In [ ]:
!python solutions/ex15_netmhcpan_iedb.py
!python solutions/ex3_mhcflurry.py data/fhaC.fasta
!python solutions/ex16_consensus.py

In [ ]:
import pandas as pd

print("Strong binder in all three class I tools:")
display(pd.read_csv("results/consensus_class_I.csv").head(12))

print("\nRegions of fhaC that carry BOTH a class I and a class II binder:")
display(pd.read_csv("results/class_I_II_overlap.csv"))

### What to notice

**27 peptide and allele pairs** are called a strong binder by NetMHCpan, MHCflurry and MixMHCpred at once. The table is sorted by the *worst* rank any tool gave them, which is the conservative way to read a consensus.

The second table is the result no slide asks for. **Four regions of FhaC carry both a class I consensus peptide and a top class II binder.** The best is residues 103 to 122, inside the POTRA domain, holding `RLFDPAPLV` for HLA-A\*02:01 and `APLVQDYLNRPLDNE` for DRB1\*15:01.

FhaC is extracellular, so class II is its natural route. A region that can feed both a CD8 and a CD4 response is a better bet than either alone.

### Discuss

1. Three tools agreeing is reassuring. Could they agree and all be wrong together? What would they have in common that would cause that?
2. We ranked by the worst rank across tools. What changes if you rank by the average, and which is the honest default?
3. MixMHCpred is not a neural network. Does that make its agreement with the other two worth more, or less?
4. A dual class I and class II region looks ideal. What experiment would you run first to test it?

---
## Exercise 12, slide 22: can deep learning predict immunogenicity?

Three claims from the slide, checked with numbers:

1. **DeepImmuno encodes** each residue as 12 PCA components of 553 AAindex properties (a 21 by 12 matrix), and the MHC enters as its paratope residues.
2. **PRIME 2.0 uses 28 inputs**: one presentation score, 20 residue frequencies at TCR facing positions, and 7 one-hot nodes for length.
3. **A strong binder is not necessarily immunogenic.**

Watch block `[12.4]`, where the AUC of the presentation score is compared against the AUC of the immunogenicity model.

> **About the split.** Training and test use different peptides, via `GroupShuffleSplit`. An ordinary random split would put the same peptide on both sides and inflate the AUC.

In [ ]:
!python solutions/ex12_immunogenicity.py

In [ ]:
from IPython.display import Image, display
display(Image("results/ex12_1_encoding.png"))
display(Image("results/ex12_4_presentation_vs_immunogenicity.png"))

In [ ]:
import pandas as pd
fhac = pd.read_csv("results/ex12_fhaC_immunogenicity.csv")
print(f"{len(fhac)} fhaC 9-mers scored")
fhac.head(15)

### Discuss

1. Why would an ordinary random split give a prettier and more dishonest AUC?
2. PRIME leaves P2 and the last position out of the frequency vector. What assumption is behind that, and does it hold for class II?
3. The slide says DeepImmuno reports low precision on SARS-CoV-2. Look at the allele composition of the training set.
4. Does a `Negative` label in IEDB mean "not immunogenic" or "nobody detected a response in that assay"?

**Open question.** The ranking above ignores whether the HLA binds the peptide at all. What do you expect to happen to this top 10 once exercise 14 adds the binding filter?

---
## Exercise 13, slide 23: does this TCR recognise this peptide?

NetTCR-2.0 rebuilt exactly as the slide draws it: BLOSUM50 zero padded to 30, 1D convolutions with 16 filters at kernel sizes 1, 3, 5, 7 and 9, max pooled into 240 values, a dense layer of 32, one sigmoid.

Then the warning in the orange box is tested:

> *The authors state the model can at present only be applied to the peptides in its training data.*

Block `[13.4]` trains **without** GILGFVFTL and evaluates **on** GILGFVFTL. Watch the AUC fall. That drop is the operational definition of extrapolation.

*This cell trains two networks and takes several minutes.*

In [ ]:
!python solutions/ex13_nettcr.py

In [ ]:
display(Image("results/ex13_limits.png"))
pd.read_csv("results/ex13_auc_per_peptide.csv")

### What to notice

The global AUC is high and the per peptide table shows why that is misleading. The dataset has **17 peptides** and three of them hold most of the data.

Block `[13.5]` has the sharpest result. On fhaC peptides, which the model never saw, the score barely moves when the peptide changes but does move when the CDR3 changes. Out of its domain the network is scoring the TCR and ignoring the peptide.

### Discuss

1. Why is the global AUC alone misleading, given the per peptide table?
2. Non binders are built by pairing TCRs with random peptides. What bias does that introduce?
3. The input has no HLA. Two people with different HLA and the same CDR3 get the same score. Is that right?
4. What experimental data would you have to generate to use NetTCR on fhaC?

---
## Exercise 14, slide 24: how do we prioritise?

Two claims from the slide, turned into code:

- **pVACtools is a pipeline, not a model.** Filters chained over predictions that already exist. Here the chain uses the UniProt topology from exercise 11, MHCflurry binding, the exercise 12 model, and the TCR column that honestly stays empty after exercise 13.
- **TIDE answers a different question.** *No peptide, no HLA allele, no epitope anywhere in this diagram.*

> Needs the model saved by exercise 12, so run that one first.

In [ ]:
!python solutions/ex14_prioritization.py

In [ ]:
display(Image("results/ex14_prioritization.png"))
shortlist = pd.read_csv("results/ex14_shortlist.csv")
print(f"shortlist: {len(shortlist)} peptide and allele pairs")
shortlist.head(20)

### Final experiment: move the weights

The combined score uses weights **we** chose, not the model. This cell changes them and measures how much the top 10 moves. It is the fastest way to see how fragile a prioritised list really is.

In [ ]:
import numpy as np

t = pd.read_csv("results/ex14_shortlist.csv")

def normalise(v, invert=False):
    v = -np.log10(np.clip(v, 1e-3, None)) if invert else np.asarray(v, float)
    return (v - v.min()) / (v.max() - v.min() + 1e-9)

nb = normalise(t.affinity_rank, invert=True)
npres = normalise(t.presentation_rank, invert=True)
ni = normalise(t.immunogenicity_score)

scenarios = {
    "balanced (the script)":   (0.35, 0.35, 0.30),
    "binding only":            (1.00, 0.00, 0.00),
    "immunogenicity only":     (0.00, 0.00, 1.00),
    "presentation heavy":      (0.20, 0.60, 0.20),
}

tops = {}
for name, (a, b, c) in scenarios.items():
    s = a * nb + b * npres + c * ni
    tops[name] = set(t.assign(s=s).nlargest(10, "s").peptide)
    print(f"{name:<26} top10: {', '.join(sorted(tops[name]))}")

base = tops["balanced (the script)"]
print("\nOverlap with the script ranking:")
for name, s in tops.items():
    print(f"  {name:<26} {len(base & s)}/10")
print("\nPeptides surviving every scenario:", len(set.intersection(*tops.values())))

### Discuss

1. Given the overlap above, what does "prioritised list" mean if it moves this much with the weights? What would you report in a paper?
2. We filtered on the median immunogenicity. Why a median and not an absolute threshold?
3. The `TCR_evidence` column is empty on purpose. What experiment would fill it?
4. We did not use conservation across *B. pertussis* strains. Why is that the obvious next filter for a vaccine?
5. If this were a tumour neoantigen instead of fhaC, which steps would change and which would not?

---

## Closing the module

Every arrow in the workflow on slide 17 shortened the list. None of them proved anything.

- **SignalP and DeepTMHMM** said where the protein goes, not whether it is antigenic.
- **MHCflurry** gave a hypothesis about the groove, not about the cell.
- **DeepImmuno and PRIME** ranked candidates. Neither classified them.
- **NetTCR** is only valid inside its 17 peptides.
- **pVACtools** chained filters, and **TIDE** does not even answer this question.

What comes out is a **ranked list of hypotheses**, ready for the bench.

One honest caveat to carry forward: fhaC reaches the immune system as an extracellular protein, so its natural route is MHC class II and CD4 T cells. Everything in class I here is a hypothesis about cross presentation. That is why `ex2_netmhciipan.py` is not optional for this antigen.

*Next, module M03.2: from sequence to structure. AlphaFold and the shape of the peptide, MHC and TCR interface, on the same fhaC.*